<a href="https://colab.research.google.com/github/Syeda-Rb-47/Building-GlueScore--An-Interpretable-Glue-Likeness-Scoring-Pipeline/blob/main/2_Data_Cleaning_and_Standardization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORTS

In [ ]:
# Installing RDKit and importing required libraries.
!pip install -q rdkit

import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.SaltRemover import SaltRemover
from google.colab import drive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 43.7 MB/s eta 0:00:00


LOADING AND MERGING DATASETS

In [ ]:
# Loading datasets from Drive.
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Thesis Datasets"
glue   = pd.read_csv(DRIVE_DIR + "/MolGlueDB dataset.csv")
chembl = pd.read_csv(DRIVE_DIR + "/ChEMBL dataset.csv")

glue.columns   = glue.columns.str.strip()
chembl.columns = chembl.columns.str.strip()

print("Glue columns:  ", glue.columns.tolist())
print("ChEMBL columns:", chembl.columns.tolist())
print("\nGlue rows:", len(glue))
print("ChEMBL rows:", len(chembl))

Mounted at /content/drive
Glue columns:   ['Source_ID', 'Canonical_SMILES', 'StdInChIKey', 'Label']
ChEMBL columns: ['Source_ID', 'Canonical_SMILES', 'StdInChIKey', 'Label']

Glue rows: 1629
ChEMBL rows: 2875


In [ ]:
# Merging glue and background datasets.
cols     = ["Source_ID", "Canonical_SMILES", "StdInChIKey", "Label"]
all_data = pd.concat([glue[cols], chembl[cols]], ignore_index=True)

print("Merged shape:",all_data.shape)
print(all_data["Label"].value_counts())

Merged shape: (4504, 4)
Label
0    2875
1    1629
Name: count, dtype: int64


STRUCTURE CLEANING FUNCTION

In [ ]:
# Removes salts and keeps the parent molecule for each SMILES.
salt_remover = SaltRemover()

def clean_structure(smiles):
    out = {
        "Clean_SMILES": None,
        "Status":       None,
        "Salt_Removed": False,
        "Changed":      False
    }

    if pd.isna(smiles) or str(smiles).strip() == "":
        out["Status"] = "empty_smiles"
        return out

    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            out["Status"] = "invalid_smiles"
            return out

        start_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

        mol = rdMolStandardize.Cleanup(mol)
        mol = salt_remover.StripMol(mol, dontRemoveEverything=True)
        mol = rdMolStandardize.FragmentParent(mol)

        clean_smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

        out["Clean_SMILES"] = clean_smiles
        out["Status"]       = "ok"
        out["Salt_Removed"] = ("." in start_smiles and "." not in clean_smiles)
        out["Changed"]      = (start_smiles != clean_smiles)

    except Exception as e:
        out["Status"] = "error: " + str(e)

    return out

In [ ]:
# Applying the cleaning function to all molecules.
cleaned  = all_data["Canonical_SMILES"].apply(clean_structure).apply(pd.Series)
all_data = pd.concat([all_data, cleaned], axis=1)

Streaming output truncated to the last 5000 lines.
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Initializing MetalDisconnector
[15:42:27] Running MetalDisconnector
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Running LargestFragmentChooser
[15:42:27] Initializing MetalDisconnector
[15:42:27] Running MetalDisconnector
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Initializing MetalDisconnector
[15:42:27] Running MetalDisconnector
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Running LargestFragmentChooser
[15:42:27] Initializing MetalDisconnector
[15:42:27] Running MetalDisconnector
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Initializing MetalDisconnector
[15:42:27] Running MetalDisconnector
[15:42:27] Initializing Normalizer
[15:42:27] Running Normalizer
[15:42:27] Running LargestFragmentChooser
[15:42:27] Initializing MetalDisconnec

In [ ]:
# Status check for rows
print("Status counts:")
print(all_data["Status"].value_counts())
print("\nChanged counts:")
print(all_data["Changed"].value_counts())
print("\nSalt_Removed counts:")
print(all_data["Salt_Removed"].value_counts())

Status counts:
Status
ok    4504
Name: count, dtype: int64

Changed counts:
Changed
False    3544
True      960
Name: count, dtype: int64

Salt_Removed counts:
Salt_Removed
False    3547
True      957
Name: count, dtype: int64


FILTERING AND DEDUPLICATION

In [ ]:
# Keeping only valid structures
clean_data = all_data[all_data["Status"] == "ok"].copy()

print("Rows before cleaning:", len(all_data))
print("Rows after cleaning: ", len(clean_data))

# Removing salt-form duplicates within each label using Clean_SMILES.
clean_data = clean_data.drop_duplicates(subset=["Label", "Clean_SMILES"], keep="first").copy()
print("Rows after SMILES deduplication:", len(clean_data))

Rows before cleaning: 4504
Rows after cleaning:  4504
Rows after SMILES deduplication: 3750


CROSS-LABEL OVERLAP DETECTION

In [ ]:
# Use StdInChIKey for overlap detection
clean_data["Structure_Key"] = clean_data["StdInChIKey"].fillna(clean_data["Clean_SMILES"])

# Identifying and removing structures that appear in both glue and background sets.
label_count  = clean_data.groupby("Structure_Key")["Label"].nunique()
overlap_keys = label_count[label_count > 1].index.tolist()
overlap      = clean_data[clean_data["Structure_Key"].isin(overlap_keys)].copy()
no_overlap   = clean_data[~clean_data["Structure_Key"].isin(overlap_keys)].copy()

print("Overlapping structures across classes:", len(overlap_keys))
print("Rows involved in overlap:             ", len(overlap))

Overlapping structures across classes: 4
Rows involved in overlap:              8


FINAL DEDUPLICATION

In [ ]:
# Removing any remaining duplicates by Structure_Key within no_overlap.
final_data = no_overlap.drop_duplicates(subset=["Structure_Key"], keep="first").copy()

print("Final unique rows:", len(final_data))
print(final_data["Label"].value_counts())

Final unique rows: 3742
Label
0    2118
1    1624
Name: count, dtype: int64


FINALISING DATASET COLUMNS

In [ ]:
# Selecting relevant columns, renaming for consistency, and adding a Master_ID.
final_clean = final_data[["Source_ID", "Label", "Clean_SMILES", "StdInChIKey"]].copy()
final_clean = final_clean.rename(columns={
    "Clean_SMILES": "Canonical_SMILES"
})
final_clean.insert(0, "Master_ID", range(1, len(final_clean) + 1))

final_clean.head()

,Master_ID,Source_ID,Label,Canonical_SMILES,StdInChIKey
0,1,1,1,[2H]C([2H])([2H])Oc1ccc2cc(C(=O)Nc3ccc4c(c3)CN...,JGWFFKIYBDZDFI-FIBGUPNXSA-N
1,2,2,1,CC/C=C\C[C@@H]1C(=O)CC[C@@H]1CC(=O)N[C@H](C(=O...,IBZYPBGPOGJMBF-QPERPISQSA-N
2,3,3,1,C#Cc1ccc(NC(=O)OC[C@]23CN(S(=O)(=O)c4ccc(C(C)(...,SVVLXZSOWBLJBW-YUGNUMPCSA-N
3,4,4,1,C#Cc1ccc(NC(=O)OC[C@]23CN(S(=O)(=O)c4ccc(C(C)(...,LPVXGYZMYNYSRZ-GGWKSQAJSA-N
4,5,5,1,O=C1CCC(NC(=O)c2ccc([N+](=O)[O-])cc2F)C(=O)N1,CSOFOICJBYMWDE-UHFFFAOYSA-N


SUMMARY OF RESULTS

In [ ]:
# Overview of key cleaning metrics from start to final dataset.
results = pd.DataFrame({
    "Metric": [
        "Initial merged rows",
        "Rows removed (salt-form duplicates)",
        "Rows removed (cross-label overlaps)",
        "Final unique rows",
        "Final glue rows",
        "Final background rows"
    ],
    "Value": [
        len(all_data),
        len(all_data[all_data["Status"] == "ok"]) - len(all_data[all_data["Status"] == "ok"].drop_duplicates(subset=["Label", "Clean_SMILES"])),
        len(overlap),
        len(final_clean),
        (final_clean["Label"] == 1).sum(),
        (final_clean["Label"] == 0).sum()
    ]
})
results

,Metric,Value
0,Initial merged rows,4504
1,Rows removed (salt-form duplicates),754
2,Rows removed (cross-label overlaps),8
3,Final unique rows,3742
4,Final glue rows,1624
5,Final background rows,2118


SAVING FINAL DATASET TO DRIVE

In [ ]:
OUT_PATH = DRIVE_DIR + "/merged_cleaned_dataset.csv"
final_clean.to_csv(OUT_PATH, index=False)

print("Saved to:   ", OUT_PATH)
print("File exists:", os.path.exists(OUT_PATH))

Saved to:    /content/drive/MyDrive/Thesis Datasets/merged_cleaned_dataset.csv
File exists: True
